# 01 · Una imagen es un tensor
*Píxeles, canales y la forma (C, H, W); y por qué una red densa no alcanza*

Notebook del episodio 1 de **Visión: de convoluciones a ViT** ([ver la lección](https://d32vh5r3pvn9et.cloudfront.net/cursos/vision/una-imagen-es-un-tensor)). Es el mismo guion del video, en texto, con cada bloque de código para correr y su salida real.

**Mecanismo:** flor de sklearn (convertida con sips a lab/datos/fotos.npz: no hay PIL) → (427, 640, 3) uint8, píxel (210, 250) = 219, 111, 39 → tensor (3, 427, 640) /255: 819.840 números → canales R|G|B como heatmap → CIFAR-10: train (50.000, 3, 32, 32), test 10.000, 5.000 por clase → normalizar con μ = (0,491; 0,482; 0,447), σ = (0,247; 0,244; 0,262) → MLP 3.072→512→256→10: 1.707.274 parámetros, 3.072 pesos por neurona, test 53,8 % (train 76,2 %) → softmax del primer gato de test: 40 % gato, 36 % perro → píxeles mezclados (permutacion(0)): MLP reentrenado 52,9 %; MLP original con columnas de W₁ reordenadas da exactamente 53,8 % (W₁x = W₁Pᵀ·Px)

**Cómo correrlo:** con el entorno de `~/Movies/serie-vision/lab` (`cd lab && uv run --project ~/Movies/serie-alineacion/lab --group notebooks --with jupyterlab jupyter lab ../notebooks`, o en VS Code eligiendo el kernel `lab/.venv`). Todo corre offline: Gemma 2 2B sale de la caché de Hugging Face, y los entrenamientos son de juguete, con datos sintéticos.

Cada bloque de código arranca de cero (`%reset -f`): la celda de **preparación** es lo que el video no muestra, y la siguiente es exactamente el código que aparece en pantalla.

## ▶ Antes de empezar

Corré la celda de abajo una vez: baja el código de la serie y sus dependencias. Funciona en Google Colab (gratis) y en cualquier Jupyter. Para guardar tus cambios en Colab: **Archivo → Guardar una copia en Drive**.

In [ ]:
import os, subprocess, sys
SERIE = "vision"
REPO = "https://github.com/Faridmurzone/paperclase-notebooks"

def _raiz():  # carpeta del repo si ya estamos adentro
    d = os.path.abspath(os.getcwd())
    while True:
        if os.path.isfile(os.path.join(d, ".paperclase-notebooks")):
            return d
        if os.path.dirname(d) == d:
            return None
        d = os.path.dirname(d)

RAIZ = _raiz()
if RAIZ is None:
    RAIZ = os.path.abspath("paperclase-notebooks")
    if not os.path.isdir(RAIZ):
        subprocess.run(["git", "clone", "--depth", "1", REPO, RAIZ], check=True)
os.chdir(os.path.join(RAIZ, SERIE, "lab"))  # los bloques de código importan desde snippets/ y leen datos/
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "../requirements.txt"], check=True)
os.environ["HF_HUB_OFFLINE"] = "0"  # el notebook original corría sin red; acá baja los modelos de Hugging Face
os.environ["TOKENIZERS_PARALLELISM"] = "false"
print("listo:", os.getcwd())

In [1]:
import os
os.environ.setdefault("HF_HUB_OFFLINE", "1")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
import warnings; warnings.filterwarnings("ignore")

*Visión, de las convoluciones a los transformers. Episodio uno: una imagen es un tensor. En esta serie vamos a ver cómo una red neuronal mira una imagen, con el código y la matemática de verdad, y entrenando todo acá, a escala chica. Hoy, la base: qué números hay en una foto, y por qué la red más simple los usa mal.*

### El mapa de la serie

El recorrido. Hoy, los datos: qué es una imagen para PyTorch. Después, tres episodios sobre la operación central, la convolución: con un kernel que se desliza, como multiplicación de matrices, y con muchos canales. Luego redes enteras: de LeNet a AlexNet, y qué aprendió por dentro una red convolucional. Cómo se entrenan redes profundas: las conexiones residuales y la normalización. Y el cierre: el Vision Transformer, que corta la imagen en pedazos y usa atención, y cómo se conectan imágenes con texto.

Imagen = tensor → Convolución → Conv como matriz → Canales y campo → LeNet → AlexNet → Qué mira una CNN → ResNet → BatchNorm → Vision Transformer → Texto e imagen

## Parte 1 · Una foto son números

Empecemos por lo concreto: abrir una foto y mirar qué hay adentro.

### Abrir una foto

La foto es una flor que viene de ejemplo con scikit-learn; la guardamos antes como arreglo de numpy. La cargamos. Imprimimos qué es: un arreglo de numpy de cuatrocientos veintisiete filas, seiscientas cuarenta columnas y tres canales, de tipo uint8, enteros sin signo de ocho bits, de cero a doscientos cincuenta y cinco. Un píxel del centro de la flor son tres números: doscientos diecinueve de rojo, ciento once de verde y treinta y nueve de azul. Naranja. Para PyTorch lo pasamos a tensor, movemos los canales adelante con permute, y dividimos por doscientos cincuenta y cinco. Contamos cuántos números son. Queda un tensor de tres por cuatrocientos veintisiete por seiscientos cuarenta: ochocientos diecinueve mil ochocientos cuarenta números reales.

In [2]:
%reset -f
import torch; torch.set_grad_enabled(True)  # %reset no borra el modo sin gradientes de un bloque anterior
# Preparación (no aparece en el video). Snippet: lab/snippets/ep01_foto.py
__file__ = 'snippets/ep01_foto.py'  # algunos snippets importan módulos hermanos (lab/snippets/_*.py, lab/comun)
import json
import os
import sys

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import numpy as np
import torch
import torch.nn.functional as F
from _cifar import ruta_lab

FOTOS = ruta_lab("datos", "fotos.npz")

In [3]:
foto = np.load(FOTOS)["flor"]                 # la foto de ejemplo de sklearn
print(type(foto).__name__, foto.shape, foto.dtype)
print("píxel (210, 250):", foto[210, 250])    # rojo, verde, azul
x = torch.from_numpy(foto)                    # (H, W, C), enteros 0..255
x = x.permute(2, 0, 1).float() / 255          # (C, H, W), reales 0..1
print("tensor:", tuple(x.shape), x.dtype)
print("números:", x.numel())

ndarray (427, 640, 3) uint8
píxel (210, 250): [219 111  39]
tensor: (3, 427, 640) torch.float32
números: 819840


In [4]:
# Verificación extra (corrió junto con el video)
chica = F.avg_pool2d(x[None], 32)[0]          # 3 × 13 × 20, solo para la figura
lado = [[round(float(v), 2) for c in range(3) for v in list(chica[c, i]) + ([0.0] if c < 2 else [])] for i in range(chica.shape[1])]
json.dump({"canales": lado, "forma": list(chica.shape)}, open(ruta_lab("datos", "figs", "ep01_foto.json"), "w"))

### Rojo, verde y azul, por separado

Esos tres canales se pueden mirar como tres imágenes en blanco y negro. Se los llama RGB, por red, green, blue: rojo, verde y azul. Acá están, achicados treinta y dos veces para que se vea cada píxel; más brillante es más valor. El centro de la flor brilla en el rojo, bastante en el verde y casi nada en el azul: por eso se ve naranja. El fondo verde oscuro aparece apenas en el canal del medio. Una foto en color son tres matrices apiladas.

### La forma (C, H, W)

$$
\text{foto} \in \{0, \dots, 255\}^{H \times W \times 3}
$$
<small>numpy: alto, ancho, canal</small>

$$
x_{c,i,j} = \tfrac{1}{255}\,\text{foto}_{i,j,c} \in [0, 1]
$$
<small>PyTorch: canal primero</small>

$$
x \in \mathbb{R}^{C \times H \times W} = \mathbb{R}^{3 \times 427 \times 640}
$$
<small>819 840 números</small>

En símbolos. La foto es un arreglo de enteros de cero a doscientos cincuenta y cinco con tres ejes: H, el alto, W, el ancho, y tres canales. En el dibujo, el píxel del centro de la flor. x, el tensor que usa PyTorch, cambia el orden de los ejes: el canal c va primero, después la fila i y la columna j. Y lo dividimos por doscientos cincuenta y cinco para que quede entre cero y uno: las redes entrenan mejor con números chicos. En el dibujo, el mismo píxel, ahora cero coma ochenta y seis, cero coma cuarenta y cuatro, cero coma quince. Así que la foto es un elemento de R a la C por H por W: tres por cuatrocientos veintisiete por seiscientos cuarenta.

## Parte 2 · El dataset: CIFAR-10

Para entrenar necesitamos muchas imágenes con su etiqueta. Vamos a usar uno chico y clásico: CIFAR diez, por el Canadian Institute for Advanced Research, que financió su creación.

### Cargar CIFAR-10

Lo cargamos con un cargador propio que lee los archivos originales. Imprimimos las formas. Contamos cuántas imágenes hay de cada clase. Y los nombres de las clases. Por último normalizamos cada canal. Train son cincuenta mil imágenes de tres por treinta y dos por treinta y dos, en enteros de ocho bits; test, diez mil. Cinco mil por clase, todas iguales. Diez clases: avión, auto, pájaro, gato, ciervo, perro, rana, caballo, barco y camión. Y después de normalizar, media cero y desvío uno.

In [5]:
%reset -f
import torch; torch.set_grad_enabled(True)  # %reset no borra el modo sin gradientes de un bloque anterior
# Preparación (no aparece en el video). Snippet: lab/snippets/ep01_cifar.py
__file__ = 'snippets/ep01_cifar.py'  # algunos snippets importan módulos hermanos (lab/snippets/_*.py, lab/comun)
import json
import os
import sys

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from _cifar import CLASES, cargar, normalizar, ruta_lab

In [6]:
X, y = cargar("train")                  # los pickles originales
Xt, yt = cargar("test")
print("train:", tuple(X.shape), X.dtype, "· test:", len(Xt))
print("por clase:", y.bincount().tolist()[:4], "...")
print("clases:", ", ".join(CLASES[:5]))
print("       ", ", ".join(CLASES[5:]))
x = normalizar(X)                       # por canal
print(f"normalizado: media {x.mean():.2f}, desvío {x.std():.2f}")

train: (50000, 3, 32, 32) torch.uint8 · test: 10000
por clase: [5000, 5000, 5000, 5000] ...
clases: avión, auto, pájaro, gato, ciervo
        perro, rana, caballo, barco, camión
normalizado: media -0.00, desvío 1.00


In [7]:
# Verificación extra (corrió junto con el video)
ej = []
for k, i in enumerate([int((y == c).nonzero()[0]) for c in (3, 7)]):  # gato, auto, caballo
    g = (X[i].float() / 255 * __import__("torch").tensor([0.299, 0.587, 0.114]).view(3, 1, 1)).sum(0)
    ej.append(g)
filas = [[round(float(v), 2) for k, g in enumerate(ej) for v in list(g[r]) + ([0.0, 0.0] if k < 1 else [])] for r in range(32)]
json.dump({"ejemplos": filas, "clases": ["gato", "caballo"]}, open(ruta_lab("datos", "figs", "ep01_cifar.json"), "w"))

### Dos imágenes de CIFAR-10, en gris

Así de chicas son. La primera imagen de entrenamiento de las clases gato y caballo, en gris, píxel por píxel: treinta y dos por treinta y dos. A esta resolución, hasta a una persona le cuesta. Es justamente lo que hace útil al dataset: es lo bastante difícil para que se note la diferencia entre redes, y lo bastante chico para entrenar en un rato.

> **Krizhevsky, 2009 — Learning Multiple Layers of Features from Tiny Images (página de CIFAR-10)** · [fuente](https://www.cs.toronto.edu/~kriz/cifar.html)
>
> **60 000 · 32×32 · 10** · «The CIFAR-10 dataset consists of 60000 32x32 colour images in 10 classes, with 6000 images per class.» Las clases no se pisan.

La fuente: el informe técnico de Alex Krizhevsky de dos mil nueve, y la página del dataset. Sesenta mil imágenes en color de treinta y dos por treinta y dos, en diez clases, seis mil por clase: cincuenta mil de entrenamiento y diez mil de test. Y aclaran que las clases son mutuamente excluyentes: un camión nunca está etiquetado como auto.

### Normalizar por canal

$$
\hat{x}_{c,i,j} = \frac{x_{c,i,j} - \mu_c}{\sigma_c}
$$
<small>μ: media del canal en train; σ: su desvío</small>

$$
\mu = (0{,}491;\ 0{,}482;\ 0{,}447),\quad \sigma = (0{,}247;\ 0{,}244;\ 0{,}262)
$$
<small>medidos sobre las 50 000 imágenes</small>

La normalización, en fórmula. A cada valor le restamos mu sub c, la media de su canal, y lo dividimos por sigma sub c, el desvío estándar de ese canal. La letra con sombrero, x con techito, es el valor normalizado. Las medias y los desvíos se miden una sola vez sobre las cincuenta mil imágenes de entrenamiento, y se usan iguales para test. En el dibujo, los seis números: el azul es un poco más oscuro en promedio. Así cada canal entra a la red centrado en cero y con escala uno.

## Parte 3 · Un MLP sobre los píxeles

La red más simple es un MLP, multilayer perceptron: una red densa. Probemos qué tan lejos llega con imágenes.

### Aplanar: de (3, 32, 32) a 3.072

Una red densa no recibe una imagen: recibe un vector. flatten pone los tres canales uno detrás del otro, y dentro de cada canal, fila por fila. Tomemos un píxel: canal verde, fila cinco, columna siete. Su lugar en el vector es c por mil veinticuatro, más i por treinta y dos, más j. Es la posición mil ciento noventa y uno, y tiene el mismo valor, ciento quince. Los vecinos de arriba y abajo quedaron a treinta y dos lugares de distancia.

In [8]:
%reset -f
import torch; torch.set_grad_enabled(True)  # %reset no borra el modo sin gradientes de un bloque anterior
# Preparación (no aparece en el video). Snippet: lab/snippets/ep01_aplanar.py
__file__ = 'snippets/ep01_aplanar.py'  # algunos snippets importan módulos hermanos (lab/snippets/_*.py, lab/comun)
import os
import sys

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from _cifar import cargar

X, _ = cargar("test", n=1)

In [9]:
x = X[0]                              # una imagen: (3, 32, 32)
v = x.flatten()                       # un vector de 3 · 32 · 32
print("forma:", tuple(x.shape), "→", tuple(v.shape))
c, i, j = 1, 5, 7                     # canal verde, fila 5, columna 7
k = c * 32 * 32 + i * 32 + j          # dónde cae en el vector
print(f"x[{c}, {i}, {j}] = {x[c, i, j]}   v[{k}] = {v[k]}")

forma: (3, 32, 32) → (3072,)
x[1, 5, 7] = 115   v[1191] = 115


### Aplanar y conectar todo con todo

Un MLP son capas densas, donde cada neurona está conectada con todas las de la capa anterior. La entrada es la imagen aplanada: tres por treinta y dos por treinta y dos, tres mil setenta y dos números en fila. Una hidden layer de quinientas doce neuronas. Otra de doscientas cincuenta y seis. Y una salida con diez números, uno por clase. El dibujo tiene pocas neuronas, pero la idea es esa: todo conectado con todo.

### Qué calcula cada capa

$$
h = \mathrm{ReLU}(W_1 x + b_1),\quad W_1 \in \mathbb{R}^{512 \times 3072}
$$
<small>cada fila de W₁ es una neurona</small>

$$
\mathrm{ReLU}(z) = \max(0, z)
$$
<small>los negativos se apagan</small>

$$
p = \mathrm{softmax}(W_3 h_2 + b_3)
$$
<small>10 logits → 10 probabilidades</small>

Cada capa hace dos cosas. Multiplica por una matriz y suma un vector: W sub uno tiene quinientas doce filas y tres mil setenta y dos columnas, y b sub uno es el bias, un número por neurona. En el dibujo, una fila de W sub uno por toda la imagen x da un solo número: una neurona mira los tres mil setenta y dos valores. Después aplica ReLU, rectified linear unit: deja pasar los positivos y pone en cero los negativos. En el dibujo, una rampa. h es el vector de quinientas doce activaciones; h sub dos, el de la segunda capa. Al final, softmax convierte los diez números de salida, los logits, en probabilidades que suman uno. En el dibujo, la primera imagen de test, un gato: el MLP le da cuarenta por ciento a gato y treinta y seis a perro.

### El MLP entrenado

Lo entrenamos antes, una sola vez: diez epochs con Adam, adaptive moment estimation, el optimizador más usado, sobre las cincuenta mil imágenes. Acá solo lo cargamos y lo medimos. Cargamos los pesos. Sus capas: tres lineales, capa uno, capa dos y salida. Contamos los parámetros. Cuántos pesos tiene cada neurona de la primera capa. Y la precisión en test. Un millón setecientos mil parámetros, tres mil setenta y dos por neurona, y cincuenta y tres coma ocho por ciento en test. Bastante más que el diez por ciento de tirar al azar, pero en train llega a setenta y seis: memoriza mucho y generaliza poco.

In [10]:
%reset -f
import torch; torch.set_grad_enabled(True)  # %reset no borra el modo sin gradientes de un bloque anterior
# Preparación (no aparece en el video). Snippet: lab/snippets/ep01_mlp.py
__file__ = 'snippets/ep01_mlp.py'  # algunos snippets importan módulos hermanos (lab/snippets/_*.py, lab/comun)
import os
import sys

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from _cifar import cargar, normalizar, ruta_lab
from _redes import cargar_modelo, contar_parametros, evaluar

Xt, yt = cargar("test")

In [11]:
mlp, info = cargar_modelo(ruta_lab("datos", "mlp_ep01.pt"))   # ya entrenado
print([n for n, _ in mlp.named_children()])   # capa1, capa2, salida
print(f"parámetros: {contar_parametros(mlp):,}")
print(f"capa1 por neurona: {mlp.capa1.weight.shape[1]:,} pesos")
print(f"precisión en test: {evaluar(mlp, normalizar(Xt), yt):.1%}"
      f" (train {info['precision_train']:.1%}; al azar 10%)")

['capa1', 'capa2', 'salida']
parámetros: 1,707,274
capa1 por neurona: 3,072 pesos
precisión en test: 53.8% (train 76.2%; al azar 10%)


In [12]:
# Verificación extra (corrió junto con el video)
import json
import torch
from _cifar import CLASES
with torch.no_grad():
    z = mlp(normalizar(Xt[:1]))[0]
json.dump({"clase": CLASES[int(yt[0])], "logits": [round(float(v), 2) for v in z], "probs": [round(float(v), 3) for v in z.softmax(0)]},
          open(ruta_lab("datos", "figs", "ep01_mlp.json"), "w"))

## Parte 4 · El MLP no sabe qué está al lado de qué

Ahora un experimento que muestra qué le falta. Mezclamos los píxeles.

### Un gato, y el mismo gato mezclado

Tomamos un orden al azar de los mil veinticuatro píxeles, y lo aplicamos igual a todas las imágenes, de train y de test. A la izquierda, un gato de test. A la derecha, el mismo gato con los píxeles mezclados: están todos, con su color, pero en otro lugar. Para vos es ruido. La pregunta es qué pasa si entrenamos el mismo MLP con imágenes así.

### Mezclar los píxeles

El experimento tiene dos partes. La permutación: un orden fijo de los mil veinticuatro píxeles. La aplicamos al test. Cargamos un segundo MLP, idéntico, entrenado desde cero con las imágenes mezcladas. Y la segunda parte: el índice de la mezcla, ahora sobre el vector aplanado de tres mil setenta y dos, canal por canal. Al MLP original le reordenamos las columnas de la primera capa con ese índice. El entrenado con píxeles mezclados llega a cincuenta y dos coma nueve, casi lo mismo. Y el original, con las columnas reordenadas, da exactamente cincuenta y tres coma ocho sobre las imágenes mezcladas: la misma red, punto por punto.

In [13]:
%reset -f
import torch; torch.set_grad_enabled(True)  # %reset no borra el modo sin gradientes de un bloque anterior
# Preparación (no aparece en el video). Snippet: lab/snippets/ep01_permutar.py
__file__ = 'snippets/ep01_permutar.py'  # algunos snippets importan módulos hermanos (lab/snippets/_*.py, lab/comun)
import json
import os
import sys

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import torch
from _cifar import cargar, normalizar, permutacion, permutar, ruta_lab
from _redes import cargar_modelo, evaluar

Xt, yt = cargar("test")
mlp, _ = cargar_modelo(ruta_lab("datos", "mlp_ep01.pt"))

In [14]:
perm = permutacion(semilla=0)              # un orden fijo de los 1.024 píxeles
Xp = normalizar(permutar(Xt, perm))        # misma mezcla para todas las fotos
mlp_p, _ = cargar_modelo(ruta_lab("datos", "mlp_perm_ep01.pt"))
print(f"entrenado con píxeles mezclados: {evaluar(mlp_p, Xp, yt):.1%}")

idx = torch.cat([c * 1024 + perm for c in range(3)])   # en el vector de 3.072
with torch.no_grad():
    mlp.capa1.weight[:] = mlp.capa1.weight[:, idx]   # reordenar columnas
print(f"MLP original, columnas reordenadas: {evaluar(mlp, Xp, yt):.1%}")

entrenado con píxeles mezclados: 52.9%
MLP original, columnas reordenadas: 53.8%


In [15]:
# Verificación extra (corrió junto con el video)
torch.manual_seed(0)
g = lambda img: (img.float() / 255 * torch.tensor([0.299, 0.587, 0.114]).view(3, 1, 1)).sum(0)
i = int((yt == 3).nonzero()[0])
a, b = g(Xt[i]), g(permutar(Xt[i:i + 1], perm)[0])
filas = [[round(float(v), 2) for v in list(a[r]) + [0.0, 0.0] + list(b[r])] for r in range(32)]
json.dump({"par": filas}, open(ruta_lab("datos", "figs", "ep01_permutar.json"), "w"))

### Por qué da exactamente igual

$$
x' = P\,x
$$
<small>P: matriz de permutación de 3072 × 3072</small>

$$
W_1 x = (W_1 P^\top)(P\,x) = W_1' \, x'
$$
<small>porque $P^\top P = I$</small>

$$
W_1' = W_1 P^\top
$$
<small>las mismas columnas, en otro orden</small>

La matemática es corta. Mezclar es multiplicar por P, una matriz de permutación: tiene un solo uno por fila y por columna, y ceros en el resto. En el dibujo, cada valor de x prima sale de otra posición de x. P transpuesta, P con filas y columnas intercambiadas, por P es la identidad, I, porque desmezclar después de mezclar deja todo como estaba. Entonces W sub uno x es igual a W sub uno P transpuesta, por P x. W sub uno prima son las mismas columnas en otro orden. En el dibujo, cada peso sigue multiplicando al mismo píxel que antes. Para un MLP, el orden de los píxeles no significa nada: cualquier mezcla le da lo mismo. Y eso es exactamente el problema.

### Lo que le falta a un MLP

- No usa la vecindad: un píxel no sabe quién está al lado
- 3.072 pesos por neurona, aunque mire un detalle chico
- Un gato arriba y un gato abajo son cosas distintas
- Memoriza: 76 % en train, 54 % en test

Resumamos qué le falta. No usa la vecindad: para el MLP, la imagen es una bolsa de tres mil setenta y dos números sin geometría. Cada neurona tiene tres mil setenta y dos pesos, aunque lo que tenga que detectar sea un borde chiquito en un rincón. Si el gato está arriba o abajo, son neuronas distintas las que tienen que aprenderlo: nada se reusa. Y por eso memoriza: setenta y seis por ciento en train y cincuenta y cuatro en test. La operación del próximo episodio arregla las tres cosas de una.

## Resumiendo

- Una foto: enteros 0–255 con forma (H, W, 3)
- En PyTorch: (C, H, W), dividido 255 y normalizado por canal
- CIFAR-10: 50.000 + 10.000 imágenes de 3 × 32 × 32, 10 clases
- MLP: 1,7 millones de parámetros, 53,8 % en test
- Mezclar los píxeles no le cambia nada: no usa la vecindad
- Próximo: la convolución, un kernel chico que se desliza